# Jour 2 · Neurone, loss et apprentissage

## Objectifs

- comprendre poids, biais et activation sans calcul avancé
- expliquer loss, optimiseur, batch et époque
- observer un neurone apprendre une relation simple


## Un neurone artificiel est un petit calcul

Il reçoit plusieurs nombres, multiplie chaque entrée par un poids, ajoute un biais puis applique éventuellement une fonction d'activation. Les poids représentent la contribution apprise de chaque entrée ; le biais permet de décaler la réponse.

Nous n'avons pas besoin de démontrer les dérivées pour comprendre la mécanique générale.


![Trois entrées pondérées allant vers un neurone](../ressources/illustrations/jour_02/01_neurone.png)

*Les poids et le biais sont ajustés par l’entraînement ; ils ne sont pas choisis manuellement ligne par ligne.*


![Calcul pas à pas dans un neurone artificiel](../ressources/illustrations/jour_02/01_neurone_calcul.png)

Chaque entrée `x` est multipliée par un poids `w`. Un poids positif pousse la somme dans le même sens que l’entrée ; un poids négatif agit dans le sens opposé. Le **biais** ajoute une valeur indépendante des entrées et permet de déplacer le point à partir duquel le neurone réagit.

La somme obtenue avant l’activation est parfois appelée **pré-activation**. La fonction d’activation transforme ensuite cette valeur. Dans le schéma, ReLU reçoit `0,7` et renvoie `0,7`.

Un neurone isolé reste un calcul simple. La puissance d’un réseau vient de nombreux neurones organisés en couches, dont les sorties deviennent les entrées des couches suivantes.


## Pourquoi une fonction d'activation ?

Une activation comme **ReLU** transforme la somme obtenue. ReLU conserve les valeurs positives et remplace les négatives par zéro. Sans transformations non linéaires, empiler plusieurs couches reviendrait essentiellement à construire une grande relation linéaire.

## La boucle d'apprentissage

Le réseau produit une sortie, la **loss** mesure son écart à la réponse attendue et l'**optimiseur** modifie légèrement les poids pour réduire cette erreur. Cette boucle est répétée sur de nombreux exemples.


![Boucle batch prédiction loss optimiseur et poids](../ressources/illustrations/jour_02/01_boucle_apprentissage.png)

*L’optimiseur ne connaît pas le métier ; il cherche seulement à réduire la loss définie.*


## Comment l'erreur revient vers les poids

![Étapes de la rétropropagation](../ressources/illustrations/jour_02/01_retropropagation.png)

La **rétropropagation** est la méthode qui attribue une part de l'erreur aux différents poids du réseau.

1. La **propagation avant** calcule une prédiction à partir des entrées.
2. La **loss** mesure l'écart entre cette prédiction et la bonne réponse.
3. La rétropropagation calcule, couche après couche et en sens inverse, dans quelle direction chaque poids devrait évoluer.
4. L'**optimiseur** applique de petites corrections aux poids.

Le **gradient** indique la sensibilité de la loss à un paramètre : son signe donne une direction et sa taille indique l'influence locale d'une petite modification. Il n'est pas nécessaire de calculer les dérivées à la main pour comprendre l'idée : le réseau teste sa prédiction, mesure son erreur, distribue cette erreur en arrière, puis se corrige.


## Descendre progressivement la loss

![Effet du learning rate sur la descente de la loss](../ressources/illustrations/jour_02/01_learning_rate.png)

L’optimisation ressemble à une descente dans un paysage dont l’altitude représente la loss. Le **gradient** indique la pente locale. L’optimiseur choisit une direction qui devrait réduire la loss, puis avance d’un certain pas.

Le **learning rate**, ou taux d’apprentissage, règle la taille de ce pas :

- trop petit, l’apprentissage peut être très lent ;
- adapté, il progresse régulièrement ;
- trop grand, il peut dépasser une bonne zone, osciller ou diverger.

Adam adapte une partie de cette dynamique automatiquement, mais son learning rate reste un hyperparamètre important. Une loss qui devient `NaN` ou explose peut signaler des données mal préparées ou un pas beaucoup trop grand.


## Batch, itération et époque

Il serait possible de calculer une correction après chaque exemple, mais les mises à jour seraient très instables. À l’inverse, utiliser toutes les données avant chaque correction peut demander beaucoup de mémoire.

Un **batch** est le compromis : le réseau traite un petit lot, moyenne son erreur puis met à jour ses poids. Une **itération** correspond à une mise à jour sur un batch. Une **époque** est terminée lorsque tous les exemples du train ont été parcourus une fois.

Avec 120 exemples et `batch_size=16`, une époque contient environ huit itérations. Au début de l’époque suivante, les exemples peuvent être mélangés et revus avec les nouveaux poids.


![Huit exemples répartis en batchs](../ressources/illustrations/jour_02/01_batch_epoque.png)

*Un batch précède une correction ; une époque signifie que tous les exemples du train ont été vus une fois.*


## Une expérience volontairement minuscule

Nous demandons à un seul neurone d'apprendre la relation `y = 2x + 1`. L'objectif n'est pas la performance, mais de voir que les poids partent d'une valeur initiale puis se rapprochent d'une relation utile.

### Les objets visibles dans le code

**TensorFlow** réalise les calculs numériques et l'optimisation. **Keras** est son interface de haut niveau. Un modèle `Sequential` empile les couches dans l'ordre écrit. `Dense(1)` crée ici un neurone de sortie.

La loss `mse`, ou erreur quadratique moyenne, met les écarts au carré puis calcule leur moyenne. **Adam** est l'optimiseur : il adapte progressivement la taille de ses corrections. Le `learning_rate` contrôle l'ampleur des pas ; trop grand peut rendre l'apprentissage instable, trop petit peut le rendre très lent.



In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass
plt.style.use("seaborn-v0_8-whitegrid")
print("TensorFlow", tf.__version__)

x = np.linspace(-1, 1, 120, dtype="float32")
y = (2 * x + 1).astype("float32")

neuron = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(1),
])
neuron.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.05), loss="mse")
history = neuron.fit(x, y, epochs=80, batch_size=16, verbose=0)

plt.plot(history.history["loss"])
plt.xlabel("époque")
plt.ylabel("MSE")
plt.title("La loss diminue pendant l'apprentissage")
plt.show()

weight, bias = neuron.layers[0].get_weights()
print("poids appris :", round(float(weight[0, 0]), 3))
print("biais appris :", round(float(bias[0]), 3))


## Lire le résultat et la courbe

Les poids sont initialisés avec de petites valeurs avant la première époque. Ils ne connaissent donc pas encore la relation `y = 2x + 1`.

La courbe affiche la MSE après chaque époque. Une baisse rapide au début signifie que le neurone corrige ses plus grosses erreurs. Une stabilisation indique que les améliorations deviennent petites. Une courbe parfaitement décroissante n’est pas obligatoire : avec des batchs, de légères variations peuvent apparaître.

Le poids devrait se rapprocher de 2 et le biais de 1. Ils ne seront pas nécessairement exacts : l’apprentissage procède par corrections successives et s’arrête après un nombre fini d’époques.

Une loss qui diminue sur le train prouve seulement que le modèle reproduit mieux les exemples vus. Elle ne suffit pas à démontrer qu’il généralisera à de nouvelles données. C’est pour cela que les réseaux suivants utiliseront une validation et un test.


## À vous de jouer — tester une nouvelle valeur

Demandez au neurone une prédiction pour `x = 4`. Comparez-la à la valeur théorique `2 × 4 + 1`.


In [ ]:
# Utilisez neuron.predict sur un tableau float32 de forme (1, 1).
pass
